In [4]:
%pip install yfiles_jupyter_graphs --quiet
%pip install pygraphviz --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
!git status

On branch yfiles-main
Your branch is up to date with 'origin/yfiles-main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .ipynb_checkpoints/yFiles_4FFBO-checkpoint.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore

no changes added to commit (use "git add" and/or "git commit -a")


In [19]:
!git push -u origin yfiles-main

Branch 'yfiles-main' set up to track remote branch 'yfiles-main' from 'origin'.
Everything up-to-date


In [23]:
!git --version 
!git remote -v

git version 2.34.1
origin	https://github.com/sgarnell/archive.git (fetch)
origin	https://github.com/sgarnell/archive.git (push)


In [56]:

# Stage changes
!git add yFiles_4FFBO.ipynb

# Commit the new changes
!git commit -m "Latest version of Graph Notebook"

# Push to remote
!git push origin yfiles-main


[yfiles-main 01d62b6] Latest version of Graph Notebook
 1 file changed, 44 insertions(+), 16 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 16 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 830 bytes | 830.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/sgarnell/archive.git
   2e75470..01d62b6  yfiles-main -> yfiles-main


In [30]:
# Step 1: Fetch the latest changes from GitHub
!git checkout origin/main -- filtered_output.gv
!git fetch origin
!git diff origin/main filtered_output.gv || true; [ $? -eq 0 ] && echo "🟢 No changes detected"

🟢 No changes detected


In [87]:
%pdb on
import pdb

Automatic pdb calling has been turned ON


In [36]:
import sys
print(sys.executable)
import re
from pygraphviz import AGraph
from yfiles_jupyter_graphs import GraphWidget

/opt/conda/bin/python


In [10]:
!curl -o filtered_output.gv https://raw.githubusercontent.com/sgarnell/archive/yfiles-main/filtered_output.gv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 74811  100 74811    0     0   238k      0 --:--:-- --:--:-- --:--:--  238k


In [5]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass



In [77]:
def _node_color_mapping(node):
    label = node.get("properties", {}).get("label", "")
    if "cent=" in label:
        return "red"  # neuron
    return "cyan"    # synapse or other


def _node_size_mapping(node):
    label = node.get("properties", {}).get("label", "")
    yf_lable = node.get("properties", {}).get("yf_label", "")
    if "cent=" in label:
        return (100.0, 100.0)
    if "--" in yf_lable:
        return (15.0, 15.0)
    else:
        return (60.0, 60.0)  # fallback height if neither condition matches


group_colors = {
    "FB5": "#FFCCCC",
    "DPM": "#CCFFCC",
    "APL": "#CCCCFF",
    # Add more as needed
}


def _node_parent_group_mapping(node):
    label = node.get("properties", {}).get("label", "")
    if "cent=" in label:
        group_id = label[:3]
        return group_id
    return None


color_name_to_hex = {
    'red': '#FF0000',
    'orangered3': '#CD3700',     # X11 approximation
    'orangered2': '#EE4000',     # X11 approximation
    'gold': '#FFD700',
    'chartreuse': '#7FFF00',
    'limegreen': '#32CD32',
    'lime': '#00FF00',
    'cyan3': '#00CDCD',          # X11 approximation
    'blue': '#0000FF'
}


# def _custom_edge_color_mapping(edge):
    
#     edgeColorName = edge.get("properties", {}).get("color", "")
#     return (color_name_to_hex[edgeColorName])

def _custom_factor_mapping(edge):
    penwidth = edge.get("properties", {}).get("penwidth", "")
    try:
        pw = float(penwidth)
    except (ValueError, TypeError):
        return 1.0  # fallback for missing or invalid penwidth

    # Define expected penwidth range from your dataset
    min_pw, max_pw = 1.0, 3.0  # adjust if your data has wider spread
    min_factor, max_factor = 1.0, 10.0

    # Clamp penwidth to expected range
    pw_clamped = max(min_pw, min(pw, max_pw))

    # Linear interpolation
    scale = (pw_clamped - min_pw) / (max_pw - min_pw)
    thickness = min_factor + scale * (max_factor - min_factor)

    return round(thickness, 2)


In [78]:
def _widget(graph):
    w = GraphWidget(graph=graph)
    w.set_node_color_mapping(_node_color_mapping)
    w.set_node_size_mapping(_node_size_mapping)
    w.set_node_parent_group_mapping(_node_parent_group_mapping)
    w.set_edge_color_mapping(_custom_edge_color_mapping)
    w.set_edge_thickness_factor_mapping(_custom_factor_mapping)
    w.hierarchic_layout()
    return w


In [79]:
# Step 1: Load the Graphviz file using pygraphviz
graph = AGraph("filtered_output.gv")

In [80]:
w = _widget(graph)
w

GraphWidget(layout=Layout(height='800px', width='100%'))

In [52]:
layout_result = w.hierarchic_layout()
print(type(layout_result))
print(dir(layout_result))

<class 'NoneType'>
['__bool__', '__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__']
